# Colab Pro Pipeline

Thin orchestrator over `src/` modules. Runs on Colab Pro L4 (24 GB VRAM).

**Steps:** install deps -> clone repo -> download data -> prepare -> splits -> solve -> traces -> format SFT -> train -> evaluate -> package -> submit

## Cell 1 - Install dependencies

In [ ]:
%%bash
pip install -q torch>=2.2.0 transformers>=4.45.0 peft>=0.12.0 trl>=0.12.0 \
    vllm>=0.12.0 datasets>=3.0.0 accelerate>=1.0.0 bitsandbytes>=0.44.0 \
    polars>=1.0.0 pyyaml>=6.0

## Cell 2 - Clone repo & set up paths

In [ ]:
import os, sys

# TODO: Set your actual repo URL before running
REPO_URL = 'https://github.com/YOUR_USER/homework3_llm_reasoning_finetuning.git'
assert 'YOUR_USER' not in REPO_URL, 'Set REPO_URL to your actual repo before running'
REPO_ROOT = '/content/repo'

if not os.path.exists(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

for d in ['data/splits', 'data/traces', 'data/sft', 'checkpoints', 'submissions', 'experiments/results']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'GPU: {os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip()}')

## Cell 3 - Download competition data via Kaggle API

In [ ]:
# Upload your kaggle.json or set env vars before running this cell
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -p data/

import zipfile
for zf in ['data/nvidia-nemotron-model-reasoning-challenge.zip']:
    if os.path.exists(zf):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall('data/')
        os.remove(zf)

assert os.path.exists('data/train.csv'), 'Download failed — check Kaggle API credentials'
print('Data files:', os.listdir('data/'))

## Cell 4 - Classify puzzles

In [ ]:
!python -m src.data.prepare --input data/train.csv --output data/puzzles_classified.jsonl

## Cell 5 - Create train/val split

In [ ]:
!python -m src.data.splits --input data/puzzles_classified.jsonl --output data/splits/ --val-pct 0.10 --seed 42

## Cell 6 - Run solvers

In [ ]:
import json
from src.metrics.competition import verify
from src.solvers.numeral import NumeralSolver
from src.solvers.gravity import GravitySolver
from src.solvers.unit_conversion import UnitConversionSolver
from src.solvers.cipher import CipherSolver
from src.solvers.bit_manipulation import BitManipulationSolver
from src.solvers.equation import EquationSolver
from src.solvers.cryptarithm import CryptarithmSolver

puzzles = []
with open('data/puzzles_classified.jsonl') as f:
    for line in f:
        puzzles.append(json.loads(line))

solvers = {
    'numeral': NumeralSolver(),
    'gravity': GravitySolver(),
    'unit_conversion': UnitConversionSolver(),
    'cipher': CipherSolver(),
    'bit_manipulation': BitManipulationSolver(),
    'equation_numeric_deduce': EquationSolver(),
    'equation_numeric_guess': EquationSolver(),
    'cryptarithm_deduce': CryptarithmSolver(),
    'cryptarithm_guess': CryptarithmSolver(),
}

all_results = []
cat_stats = {}
for p in puzzles:
    cat = p['category']
    solver = solvers.get(cat)
    if solver is None:
        continue
    result = solver.solve(p)
    correct = verify(p['answer'], result.predicted_answer)
    all_results.append({
        'puzzle_id': p['id'], 'category': cat,
        'predicted_answer': result.predicted_answer,
        'expected_answer': p['answer'],
        'is_correct': correct,
        'solve_method': result.solve_method,
        'confidence': result.confidence,
    })
    cat_stats.setdefault(cat, [0, 0])
    cat_stats[cat][1] += 1
    if correct:
        cat_stats[cat][0] += 1

print(f"{'Category':<30} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print('-' * 60)
tc = ta = 0
for cat in sorted(cat_stats):
    c, t = cat_stats[cat]
    tc += c; ta += t
    print(f"{cat:<30} {c:>8} {t:>8} {c/t:>10.4f}")
print('-' * 60)
print(f"{'TOTAL':<30} {tc:>8} {ta:>8} {tc/ta:>10.4f}")

## Cell 7 - Generate CoT traces

In [ ]:
import os
from src.trace_generators.numeral_traces import NumeralTraceGenerator
from src.trace_generators.gravity_traces import GravityTraceGenerator
from src.trace_generators.unit_conversion_traces import UnitConversionTraceGenerator
from src.trace_generators.cipher_traces import CipherTraceGenerator
from src.trace_generators.bit_manipulation_traces import BitManipulationTraceGenerator
from src.trace_generators.equation_traces import EquationTraceGenerator
from src.trace_generators.cryptarithm_traces import CryptarithmTraceGenerator

trace_generators = {
    'numeral': NumeralTraceGenerator(),
    'gravity': GravityTraceGenerator(),
    'unit_conversion': UnitConversionTraceGenerator(),
    'cipher': CipherTraceGenerator(),
    'bit_manipulation': BitManipulationTraceGenerator(),
    'equation_numeric_deduce': EquationTraceGenerator(),
    'equation_numeric_guess': EquationTraceGenerator(),
    'cryptarithm_deduce': CryptarithmTraceGenerator(),
    'cryptarithm_guess': CryptarithmTraceGenerator(),
}

result_lookup = {r['puzzle_id']: r for r in all_results if r['is_correct']}
puzzle_lookup = {p['id']: p for p in puzzles}

traces_by_cat = {}
for puzzle_id, r in result_lookup.items():
    cat = r['category']
    gen = trace_generators.get(cat)
    if gen is None:
        continue
    puzzle = puzzle_lookup[puzzle_id]
    solver_result = solvers[cat].solve(puzzle)
    try:
        trace = gen.generate_trace(puzzle, solver_result)
        if trace.final_answer:
            traces_by_cat.setdefault(cat, []).append({
                'puzzle_id': trace.puzzle_id,
                'category': trace.category,
                'thinking_text': trace.thinking_text,
                'final_answer': trace.final_answer,
                'token_count': trace.token_count,
                'is_verified': trace.is_verified,
            })
    except Exception as e:
        pass

# Save traces (clear old files first)
total_traces = 0
written_files = set()
for cat, traces in traces_by_cat.items():
    base_cat = cat.replace('_deduce', '').replace('_guess', '').replace('_numeric', '')
    outfile = f'data/traces/{base_cat}_traces.jsonl'
    mode = 'a' if outfile in written_files else 'w'
    written_files.add(outfile)
    with open(outfile, mode) as f:
        for t in traces:
            f.write(json.dumps(t) + '\n')
    total_traces += len(traces)
    print(f'{cat}: {len(traces)} traces')

print(f'\nTotal traces generated: {total_traces}')

## Cell 8 - Format SFT data

In [ ]:
!python -m src.data.format_sft \
    --traces data/traces/ \
    --puzzles data/puzzles_classified.jsonl \
    --split data/splits/train_ids.json \
    --max-tokens 7200 \
    --output data/sft/train_sft_full.jsonl

## Cell 9 - QLoRA fine-tuning

In [ ]:
!python -m src.train --config experiments/configs/exp-011-full-sft.yaml

## Cell 10 - Evaluate fine-tuned model (vLLM)

In [ ]:
# Free training model VRAM before loading vLLM
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
!python -m src.evaluate \
    --adapter checkpoints/exp-011-full-sft \
    --config experiments/configs/exp-011-full-sft.yaml \
    --output experiments/results/exp-011.json

## Cell 11 - Package submission

In [ ]:
!python -m src.package \
    --adapter checkpoints/exp-011-full-sft \
    --output submissions/exp-011.zip

import zipfile, json
with zipfile.ZipFile('submissions/exp-011.zip', 'r') as z:
    print('Submission contents:')
    for info in z.infolist():
        print(f'  {info.filename}: {info.file_size/1024:.1f} KB')
    with z.open('adapter_config.json') as f:
        cfg = json.load(f)
    print(f'\nAdapter rank: {cfg.get("r", "?")}')
    print(f'Target modules: {cfg.get("target_modules", "?")}')

## Cell 12 - Submit to Kaggle

In [ ]:
!kaggle competitions submit \
    -c nvidia-nemotron-model-reasoning-challenge \
    -f submissions/exp-011.zip \
    -m "exp-011: full SFT on all categories, gradient checkpointing, max_tokens=7200"